In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Delegate to another team's agent over A2A

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

Large organisations have many agent teams. The store operations team owns its agent; a regional operations team builds its own assistant and wants to ask the store agent questions. Importing another team's tools and prompts would tie the two together. Instead, one agent calls the other over a protocol.

### The Agent2Agent (A2A) protocol

[A2A](https://a2a-protocol.org/) is an open protocol for agents to talk to each other. An agent publishes an **agent card**, a JSON description of what it does and where to reach it, and accepts tasks over HTTP. The calling agent needs only the card URL.

### A2A in ADK

`adk api_server --a2a` serves an existing ADK app over A2A, next to a card file. On the calling side, [`RemoteA2aAgent`](https://adk.dev/a2a/) turns a card into a sub-agent: the coordinator transfers to it like any other sub-agent, and the remote team's tools, prompts and guardrails stay on their side.

<img width="60%" src="../../docs/diagrams/q12.png" alt="The regional desk agent transfers to the store operations agent through its A2A agent card" />

### Objectives

In this tutorial, you will learn how to call an agent that another team owns, knowing only its agent card.

You will complete the following tasks:

- Serve the store operations app over A2A
- Read its agent card
- Build the regional desk agent from the card and ask it a store question
- Stop the server

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded in the workshop notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"
os.environ["STORE_OPS_PREWARM"] = "0"

# The quickstart imports shared store code from the repository root, two folders up
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agents" / "cymbal_store_ops").is_dir())
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the output to the agent's own events: ADK marks experimental and deprecated features
# with warnings and logs configuration hints, and the Gen AI SDK logs a note whenever a
# response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
import json
import subprocess
import time
import urllib.request

from google.adk.runners import InMemoryRunner
from google.genai import types

## Serve the store agent over A2A

### The card

The store operations team publishes its app from `server/cymbal_store_ops/`. That folder only re-exports the app from `agents/cymbal_store_ops` and adds `agent.json`, the card. Read it:

In [5]:
card = json.loads(Path("server/cymbal_store_ops/agent.json").read_text())

print(card["name"], "|", card["description"])
for skill in card.get("skills", []):
    print(f"  skill: {skill['name']}")

cymbal_store_ops | Cymbal Beauty store operations assistant for store managers and associates: start-of-day priorities, on-shelf availability, BOPIS coverage, shrink patterns, coaching summaries and store tasks with confirmation. The caller names the employee it acts for with a demo id such as U-M014.
  skill: On-shelf availability
  skill: Start-of-day plan
  skill: BOPIS coverage


### Start the server

In production the store team runs this on its own infrastructure. Here you start it as a background process on port 8002, with the same environment as the notebook. The server takes a few seconds to load the app.

In [6]:
A2A_PORT = 8002
A2A_BASE = f"http://localhost:{A2A_PORT}"
CARD_URL = f"{A2A_BASE}/a2a/cymbal_store_ops/.well-known/agent-card.json"

server = subprocess.Popen(
    [sys.executable, "-m", "google.adk.cli", "api_server", "--a2a", "server", "--port", str(A2A_PORT)],
    env={**os.environ, "PYTHONPATH": str(REPO_ROOT)},
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(60):
    try:
        with urllib.request.urlopen(CARD_URL) as response:
            served_card = json.load(response)
        break
    except OSError:
        time.sleep(1)
else:
    server.terminate()
    raise RuntimeError(f"The A2A server did not publish {CARD_URL} within 60 seconds")

print(f"Serving {served_card['name']} at {served_card['supportedInterfaces'][0]['url']}")

Serving cymbal_store_ops at http://localhost:8002/a2a/cymbal_store_ops


## Build the calling agent

`agent.py` in this folder is the regional desk. Its only sub-agent is a `RemoteA2aAgent` built from the card URL; nothing from the store operations code is imported. It reads the base URL from `STORE_OPS_A2A_URL`, which defaults to `localhost:8002`.

In [7]:
os.environ["STORE_OPS_A2A_URL"] = A2A_BASE

from agent import app

desk = app.root_agent
for sub_agent in desk.sub_agents:
    print(f"{type(sub_agent).__name__}: {sub_agent.name}\n  {sub_agent.description}")

RemoteA2aAgent: cymbal_store_ops_remote
  The Cymbal Beauty store operations app, reached over A2A: a store's start-of-day priorities, on-shelf availability, BOPIS coverage, shrink, coaching and store tasks.


## Ask a store question

Start a session and define a helper that prints each transfer and the answer. The regional desk has no store tools, so it transfers the question to the store agent and relays the answer. The employee id travels in the message, because the store agent signs the caller in with it.

In [8]:
runner = InMemoryRunner(app=app)
session = await runner.session_service.create_session(app_name=app.name, user_id="regional-desk")

In [9]:
async def ask(question: str) -> None:
    """Send one message and print the tool calls, transfers and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            if call.name == "transfer_to_agent":
                print(f"[{event.author}] hands over to {call.args.get('agent_name')}")
            else:
                print(f"[{event.author}] calls {call.name}({dict(call.args or {})})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n[{event.author}] {text}\n")

In [10]:
await ask("I'm U-M014, the Naperville store manager. Why is Lumière Hydra Cream flagged?")

[a2a_agent] hands over to cymbal_store_ops_remote



[cymbal_store_ops_remote] Lumière Hydra Cream (P-0101) is flagged for an empty shelf with store stock and for being below its reorder point:

* **Empty shelf**: 0 units are on the sales floor (fixture SK-04), while all 7 on-hand units are located in backstock (bay B2).
* **Below reorder point**: Total on-hand is 7 units against a reorder threshold of 12, and an inbound shipment of 12 units is currently delayed from October 1.
* **Pending demand**: 4 units are actively reserved for 3 pickup orders due between 9:30 AM and 10:00 AM, leaving 3 units available to replenish to the sales floor.



The question takes about 30 seconds, because the full store agent runs in the server behind the transfer. The answer names P-0101: 0 units on the shelf, all 7 in the backroom, and 7 on hand against a reorder point of 12. The wording and the extra detail differ from run to run.

The transfer and the answer come back over HTTP. Everything that happened inside the store agent (its tools, its specialists, its guardrails) ran in the server process, owned by the other team.

## Cleaning up

Stop the A2A server.

In [11]:
server.terminate()
server.wait(timeout=10)
print("A2A server stopped")

A2A server stopped


## What's next

- [A2A in ADK](https://adk.dev/a2a/)
- [Agent Registry](https://docs.cloud.google.com/agent-registry/overview), where teams publish agent cards so others can find them
- [Quickstart guide](README.md) for running this agent in the ADK developer UI
- [All quickstarts](../README.md)